<table class="table table-bordered">
    <tr>
        <th style="text-align:center; vertical-align: middle; width:50%"><img src='https://www.np.edu.sg/images/default-source/default-album/img-logo.png'"></th>
        <th style="text-align:center;"><h1>Deep Learning</h1><h2>Assignment 2 (Problem 1) - Sentiment Analysis Model  (Group)</h2><h3>AY2023/24 Semester</h3></th>
    </tr>
</table>

In [4]:
# Import the Required Packages
import os
import nltk
import tensorflow as tf
import numpy as np
import pandas as pd
import contractions
import emoji
import re

from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [5]:
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

In [6]:
# Remove this if not required
print(tf.config.list_physical_devices()) 
 
# Check if GPU is available 
gpus = tf.config.list_physical_devices('GPU') 
if gpus: 
    print(f"GPUs available: {gpus}") 
else: 
    print("No GPUs available.")

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Step 1 – Data Loading and Processing (Group)

### 1.1 Data Loading

In [7]:
# Load dataset
base_dir = os.getcwd()
file_location = base_dir + '/netflix_reviews200k.csv'
netflix_review = pd.read_csv(file_location)

In [8]:
netflix_review.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,1255bae6-a139-4b71-a750-193ddf177f10,Jacob Eddy,https://play-lh.googleusercontent.com/a/ACg8oc...,Suddenly saying my device is not part of the f...,1,0,8.142.1 build 13 51334,2025-01-20 13:46:43,NaN,NaN,8.142.1 build 13 51334
1,0e76ddd6-2839-4a73-a436-6e3138da1c58,Mehedi Hasan,https://play-lh.googleusercontent.com/a/ACg8oc...,With Right Password.. I Can't Login My Account...,1,0,8.131.0 build 3 50829,2025-01-20 13:45:41,NaN,NaN,8.131.0 build 3 50829
2,7fead878-dd7e-42cb-8e85-c0d0cb100e71,Jennifer Ibama,https://play-lh.googleusercontent.com/a-/ALV-U...,network issues,5,0,8.142.1 build 13 51334,2025-01-20 13:37:47,NaN,NaN,8.142.1 build 13 51334
3,6b47ec27-09c2-4dca-bcb7-1965de35c814,Zameer Abbas,https://play-lh.googleusercontent.com/a-/ALV-U...,Nice,5,0,8.141.1 build 13 51230,2025-01-20 13:31:37,NaN,NaN,8.141.1 build 13 51230
4,6f8c5b47-8faf-472a-90d2-2ced3ba1e22d,Eiann Gracee Go,https://play-lh.googleusercontent.com/a-/ALV-U...,I really like this movie I can watch anything ...,5,0,8.142.1 build 13 51334,2025-01-20 13:22:11,NaN,NaN,8.142.1 build 13 51334
5,7d0a9872-f4a1-41c3-ac25-8a08218c5978,Yusuf Rehan,https://play-lh.googleusercontent.com/a-/ALV-U...,Booooooo,5,0,8.142.1 build 13 51334,2025-01-20 13:19:55,NaN,NaN,8.142.1 build 13 51334
6,6080a574-e5e5-4e4c-ad84-6265ed97572d,Arnold Abrenzosa Minoro,https://play-lh.googleusercontent.com/a-/ALV-U...,"Recently I downloaded movies, but when i play ...",2,0,NaN,2025-01-20 13:16:59,NaN,NaN,NaN
7,5f40ebeb-a00e-4e49-a963-bba92e7c291a,Hazel Viagedor,https://play-lh.googleusercontent.com/a/ACg8oc...,I can't open my account for several days now.,1,0,8.142.1 build 13 51334,2025-01-20 13:14:39,NaN,NaN,8.142.1 build 13 51334
8,7a01165b-de93-4590-ba55-411622544f9c,Taonana Romeo,https://play-lh.googleusercontent.com/a/ACg8oc...,I Love it,5,0,NaN,2025-01-20 12:37:58,NaN,NaN,NaN
9,9da2ff7b-de1c-4bf1-a924-540da1951bdb,Tina McGlothlin,https://play-lh.googleusercontent.com/a/ACg8oc...,It duse say I'm not apart of family some times...,5,0,8.142.1 build 13 51334,2025-01-20 12:34:13,NaN,NaN,8.142.1 build 13 51334


In [ ]:
netflix_review.drop(columns=['reviewId','userName','userImage',
                                           'thumbsUpCount', 'reviewCreatedVersion', 
                                           'at', 'replyContent', 'repliedAt', 'appVersion'], inplace=True)

In [ ]:
netflix_review.head()

In [ ]:
# Extract content and scores
texts = data['content'].astype(str)  # Reviews
labels = data['score']  # Scores

### 1.2 Data Processing


In [ ]:
# Convert the content and scores into numeric tensors
# Convert scores to categories (optional, for classification)
labels = labels - 1  # If scores are 1-5, shift to 0-4 for categorical labels

# Tokenize the texts
tokenizer = Tokenizer(num_words=10000)  # Use top 10,000 words
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
word_index = tokenizer.word_index

# Pad sequences
maxlen = 100  # Limit to 100 words per review
X = pad_sequences(sequences, maxlen=maxlen)

# One-hot encode labels (for classification tasks)
from tensorflow.keras.utils import to_categorical
y = to_categorical(labels, num_classes=5)  # Adjust based on number of score categories

### 1.3 Data Sampling

In [ ]:
# Split the X & y into train and test sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Step 2 – Develop a Sentiment Analysis Model (Individual: One Model Per Student)

### Model #1  (Student Name: XXXX)

In [ ]:
# Build the Model
model = Sequential([
    Embedding(input_dim=10000, output_dim=128, input_length=maxlen),
    LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(5, activation='softmax')  # Adjust units for classification (e.g., 5 for 1-5 scores)
])

model.compile(optimizer='adam', 
              loss='categorical_crossentropy',  # Use 'mse' for regression tasks
              metrics=['accuracy'])

model.summary()

In [ ]:
# Train the Model
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=10,
    validation_split=0.2
)

In [ ]:
# Plot the Training and Validation Accuracy & Loss Scores
import matplotlib.pyplot as plt
acc = history.history['acc']
val_acc = history.history['val_acc']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(len(acc))

plt.plot(epochs, acc, 'bo', label='Training acc')
plt.plot(epochs, val_acc, 'b', label='Validation acc')
plt.title('Training and validation accuracy')
plt.legend()
plt.savefig('graphs/model0_a')

plt.figure()

plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend()
plt.savefig('graphs/model0_l')

plt.show()

In [ ]:
# Save the Model
model.save('text_model_1.keras')

### Model #2  (Student Name: XXXX)

In [ ]:
# Build the Model


In [ ]:
# Train the Model


In [ ]:
# Plot the Training and Validation Accuracy & Loss Scores


### Model #3  (Student Name: XXXX)

In [ ]:
# Build the Model


In [ ]:
# Train the Model


In [ ]:
# Plot the Training and Validation Accuracy & Loss Scores


In [ ]:
# Save the Model
model.save('text_model_2.keras')

### Model #4  (Student Name: XXXX)

In [ ]:
# Build the Model


In [ ]:
# Train the Model


In [ ]:
# Plot the Training and Validation Accuracy & Loss Scores


In [ ]:
# Save the Model
model.save('text_model_2.keras')

### Model #5  (Student Name: XXXX)

In [ ]:
# Build the Model


In [ ]:
# Train the Model


In [ ]:
# Plot the Training and Validation Accuracy & Loss Scores


In [ ]:
# Save the Model
model.save('text_model_2.keras')

## Step 3 – Evaluate the Model using Testing Data (Individual & Group)

In [ ]:
# Model #1 (replicate where necessary for other models)
model1 = keras.models.load_model('text_model_1.keras')


In [ ]:
# Model #2 (replicate where necessary for other models)
model2 = keras.models.load_model('text_model_2.keras')


In [ ]:
# Model #3 (replicate where necessary for other models)
model3 = keras.models.load_model('text_model_3.keras')


In [ ]:
# Model #4 (replicate where necessary for other models)
model4 = keras.models.load_model('text_model_4.keras')


In [ ]:
# Model #5 (replicate where necessary for other models)
model5 = keras.models.load_model('text_model_5.keras')


In [ ]:
# Save the Best Model
model.save('text_model_best.keras')

## Step 4 – Use the Best Model to make prediction (Group)

In [ ]:
best_model = keras.models.load_model('text_model_best.keras')

In [ ]:
# takes the user input
text_input = np.array([input()])

In [ ]:
# convert the user input into numeric tensor


In [ ]:
# show the model output using predict function

